# Word2Vec

## Setup and Imports

In [1]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.io as pio
import matplotlib.pyplot as plt
from gensim.corpora import Dictionary
from gensim.models import word2vec

from sklearn.manifold import TSNE as tsne

In [2]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

In [3]:
import gensim
gensim.__version__

'4.3.3'

In [4]:
OHCO = ['doc_title','para_num','sentence_num','token_num']
BAG = OHCO[:3]


In [5]:
TOKENS = pd.read_csv('data/p2591-TOKENS.csv').set_index(OHCO).dropna() # Dropped nan tokens because it was causing errors in embeding
TOKENS.head()

pos_tuple pos token_str  \
doc_title para_num sentence_num token_num                                 
ASHPUTTEL 0        0            0           ('The', 'DT')  DT       The   
                                1          ('wife', 'NN')  NN      wife   
                                2            ('of', 'IN')  IN        of   
                                3             ('a', 'DT')  DT         a   
                                4          ('rich', 'JJ')  JJ      rich   

                                          term_str pos_group  
doc_title para_num sentence_num token_num                     
ASHPUTTEL 0        0            0              the        DT  
                                1             wife        NN  
                                2               of        IN  
                                3                a        DT  
                                4             rich        JJ

## Convert to Gensim

In [6]:
docs = TOKENS.groupby(BAG).term_str.apply(list).tolist()

In [7]:
for i in range(5):
    print(f"Doc {i}:", docs[i])

Doc 0: ['the', 'wife', 'of', 'a', 'rich', 'man', 'fell', 'sick']
Doc 1: ['and', 'when', 'she', 'felt', 'that', 'her', 'end', 'drew', 'nigh', 'she', 'called', 'her', 'only', 'daughter', 'to', 'her', 'bedside', 'and', 'said', 'always', 'be', 'a', 'good', 'girl', 'and', 'i', 'will', 'look', 'down', 'from', 'heaven', 'and', 'watch', 'over', 'you']
Doc 2: ['soon', 'afterwards', 'she', 'shut', 'her', 'eyes', 'and', 'died', 'and', 'was', 'buried', 'in', 'the', 'garden']
Doc 3: ['and', 'the', 'little', 'girl', 'went', 'every', 'day', 'to', 'her', 'grave', 'and', 'wept', 'and', 'was', 'always', 'good', 'and', 'kind', 'to', 'all', 'about', 'her']
Doc 4: ['and', 'the', 'snow', 'fell', 'and', 'spread', 'a', 'beautiful', 'white', 'covering', 'over', 'the', 'grave']


In [8]:
dictionary = Dictionary(docs) 


## Generate Embeddings

In [9]:
w2v_params = dict(
    window = 2,
    vector_size = 200,
    min_count = 50, 
    workers = 4
)

In [10]:
model = word2vec.Word2Vec(docs, **w2v_params)
model.wv.vectors

array([[-0.12487973,  0.17795187, -0.10016053, ..., -0.20993258,
        -0.19321428, -0.11712994],
       [-0.09554478,  0.12292265, -0.10152432, ..., -0.23765153,
        -0.13589194, -0.13159534],
       [-0.01216235,  0.01817682, -0.08385931, ..., -0.22164376,
        -0.02945091, -0.09067033],
       ...,
       [-0.01433955,  0.02654687, -0.07834896, ..., -0.21899444,
        -0.04241911, -0.08841863],
       [-0.00743989,  0.01636544, -0.07581241, ..., -0.20477591,
        -0.04510811, -0.08159964],
       [ 0.02483428, -0.02658961, -0.08101919, ..., -0.21281357,
        -0.01321515, -0.07135378]], dtype=float32)

In [11]:
WV = pd.DataFrame(model.wv.vectors, index=model.wv.index_to_key)
WV.index.name = 'term_str'
WV

,0,1,2,3,4,5,6,7,8,9,...,190,191,192,193,194,195,196,197,198,199
term_str,,,,,,,,,,,,,,,,,,,,,
the,-0.124880,0.177952,-0.100161,0.049827,0.007597,0.044845,0.172225,0.246521,-0.096341,0.155641,...,0.118092,0.022944,-0.099423,-0.217131,0.201795,0.022315,0.135784,-0.209933,-0.193214,-0.117130
and,-0.095545,0.122923,-0.101524,0.061160,0.049754,0.022025,0.125008,0.217032,-0.096295,0.121668,...,0.106194,0.026160,-0.133482,-0.205090,0.194935,0.006975,0.131123,-0.237652,-0.135892,-0.131595
to,-0.012162,0.018177,-0.083859,0.132648,0.126807,-0.060902,0.117477,0.224857,-0.099356,0.097564,...,0.089242,0.027622,-0.102750,-0.128657,0.125481,-0.009639,0.105371,-0.221644,-0.029451,-0.090670
he,-0.006998,0.044136,-0.114013,0.117239,0.113533,-0.035201,0.156287,0.223869,-0.079570,0.099774,...,0.119004,0.000293,-0.095842,-0.113418,0.157520,0.033554,0.090563,-0.230489,-0.050572,-0.117515
a,-0.055081,0.076579,-0.070601,0.128057,0.110410,-0.029403,0.157470,0.224132,-0.112186,0.117031,...,0.097385,0.019282,-0.099323,-0.130401,0.165590,-0.004629,0.107326,-0.219381,-0.067264,-0.104990
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
wished,-0.000303,-0.000470,-0.072895,0.138875,0.128388,-0.065008,0.109079,0.212913,-0.082088,0.083194,...,0.068954,0.014529,-0.080486,-0.103538,0.100651,-0.004371,0.094858,-0.193912,-0.033399,-0.085022
better,0.033784,-0.033234,-0.080120,0.183972,0.178701,-0.110339,0.122679,0.238298,-0.095577,0.074036,...,0.065390,0.007623,-0.078783,-0.085898,0.085470,-0.002725,0.106274,-0.221021,-0.002095,-0.079384
third,-0.014340,0.026547,-0.078349,0.126826,0.120886,-0.052591,0.121031,0.220649,-0.085873,0.093236,...,0.085366,0.005601,-0.092285,-0.122784,0.129031,0.003046,0.101269,-0.218994,-0.042419,-0.088419


## Make and Plot TSNE

In [12]:
PP = 50 #40 # Try 1, 100, etc.

tsne_engine = tsne(
    perplexity=PP, 
    n_components=2, 
    init='pca', 
    max_iter=2500, 
    random_state=23
)
TSNE = pd.DataFrame(
    tsne_engine.fit_transform(WV), 
    columns=['x','y'], 
    index=WV.index)
TSNE

,x,y
term_str,,
the,14.060517,0.249429
and,13.241228,0.351034
to,-1.499396,-1.659802
he,8.631740,-3.451618
a,9.339996,-1.494647
...,...,...
wished,-3.756106,1.675403
better,-9.278430,-0.198421
third,1.524298,-0.391691


In [20]:
!pip install kaleido


  Using cached kaleido-1.2.0-py3-none-any.whl.metadata (5.6 kB)
  Using cached choreographer-1.2.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached logistro-2.0.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached pytest_timeout-2.4.0-py3-none-any.whl.metadata (20 kB)
  Using cached pytest-9.0.3-py3-none-any.whl.metadata (7.6 kB)
  Using cached iniconfig-2.3.0-py3-none-any.whl.metadata (2.5 kB)
  Using cached pluggy-1.6.0-py3-none-any.whl.metadata (4.8 kB)
Using cached kaleido-1.2.0-py3-none-any.whl (68 kB)
Using cached choreographer-1.2.1-py3-none-any.whl (49 kB)
Using cached logistro-2.0.1-py3-none-any.whl (8.6 kB)
Using cached pytest_timeout-2.4.0-py3-none-any.whl (14 kB)
Using cached pytest-9.0.3-py3-none-any.whl (375 kB)
Using cached pluggy-1.6.0-py3-none-any.whl (20 kB)
Using cached iniconfig-2.3.0-py3-none-any.whl (7.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [kaleido]6/10 [pytest]


In [21]:
fig=px.scatter(TSNE.reset_index(), 'x', 'y', 
        text='term_str', 
        hover_name='term_str',  
        # size='n',
        height=1000,
        width=1200)\
    .update_traces(
        mode='markers+text', 
        textfont=dict(color='black', size=14, family='Arial'),
        textposition='top center')
fig.write_image('output/p2591-WORD2VEC.png')
fig.show()

ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido


## Save Outputs

In [ ]:
WV.to_csv('output/p2591-WORD2VEC.csv')
